In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'WenQuanYi Zen Hei'
# -------------------------- 核心配置 --------------------------
# 选择你要处理的图片路径（替换成自己的图片！）
INPUT_IMAGE_PATH = "labuduo.png"  
# 输出图片保存路径
OUTPUT_IMAGE_PATH = "deepdream_dog.png"

# -------------------------- 加载预训练模型 --------------------------
# 使用 Google 经典的 InceptionV3（DeepDream 原生模型）
base_model = tf.keras.applications.InceptionV3(
    include_top=False,  # 去掉分类层，只保留特征提取
    weights="imagenet"  # 自动下载预训练权重
)

# 选择要放大的神经网络层（多层组合效果更梦幻）
layers = [
    base_model.get_layer("mixed3").output,
    base_model.get_layer("mixed5").output,
]
# 构建 DeepDream 模型：输入图像 → 输出选定层的特征
dream_model = tf.keras.Model(inputs=base_model.input, outputs=layers)

# -------------------------- 核心算法：梯度上升 --------------------------
def calc_loss(image, model):
    """计算损失：特征激活值越大，损失越高（我们要最大化它）"""
    img_batch = tf.expand_dims(image, axis=0)  # 增加批次维度
    layer_activations = model(img_batch)
    loss = tf.reduce_mean([tf.reduce_mean(act) for act in layer_activations])
    return loss

@tf.function  # 加速计算
def dream_step(image, model, learning_rate):
    """单步梦境优化：计算梯度 → 更新图像"""
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = calc_loss(image, model)
    # 计算损失对图像的梯度
    gradients = tape.gradient(loss, image)
    # 标准化梯度（稳定优化）
    gradients /= tf.math.reduce_std(gradients) + 1e-8
    # 梯度上升：更新图像（放大特征）
    image = image + gradients * learning_rate
    # 限制图像像素值在合法范围
    image = tf.clip_by_value(image, -1, 1)
    return loss, image

# -------------------------- 图像预处理/后处理 --------------------------
def preprocess_image(image_path):
    """读取图片 → 标准化为模型输入格式"""
    img = Image.open(image_path).convert("RGB")
    img = np.array(img)
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return img

def deprocess_image(img):
    """将模型输出还原为普通图片"""
    img = 255 * (img + 1.0) / 2.0
    return Image.fromarray(tf.cast(img, tf.uint8).numpy())

# -------------------------- 主函数：生成 DeepDream 图片 --------------------------
def deep_dream(image, steps=50, step_size=0.01):
    """迭代生成梦境图片（类似遗传算法的迭代进化）"""
    for step in range(steps):
        loss, image = dream_step(image, dream_model, step_size)
        # 每10步打印进度
        if step % 10 == 0:
            print(f"迭代次数：{step:>3} | 损失值：{loss.numpy():.4f}")
    return image

# -------------------------- 运行！--------------------------
if __name__ == "__main__":
    # 1. 加载并预处理原图
    original_img = preprocess_image(INPUT_IMAGE_PATH)
    
    # 2. 生成梦境图片
    dream_img = deep_dream(original_img, steps=250, step_size=0.01)
    
    # 3. 还原并保存结果
    result = deprocess_image(dream_img)
    result.save(OUTPUT_IMAGE_PATH)
    
    # 4. 显示对比图
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.title("原始图片")
    plt.imshow(deprocess_image(original_img))
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.title("DeepDream 梦境图片")
    plt.imshow(result)
    plt.axis("off")
    plt.show()


In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random
plt.rcParams['font.family'] = 'WenQuanYi Zen Hei'
# ===================== 疯狂模式核心配置（可按需微调）=====================
INPUT_IMAGE_PATH = "fangzi.jpg"   # 你的输入图片路径
OUTPUT_IMAGE_PATH = "crazy_deepdream_fangzi.jpg"  # 疯狂效果输出路径
OCTAVE_COUNT = 6  # 八度缩放数（越大越疯狂，推荐3-6，6是极致）
STEPS_PER_OCTAVE = 200  # 每八度迭代次数（越大纹理越密集）
STEP_SIZE = 0.03  # 步长（越大变化越剧烈，0.01-0.03为宜）
GRADIENT_POWER = 1.5  # 梯度增强指数（越大纹理越突出）
RANDOM_SHIFT = True  # 随机偏移（开启更混乱）
RANDOM_SCALE = (0.95, 1.05)  # 随机缩放范围（开启让纹理更无规律）

# ===================== 加载预训练模型（全特征层激活）=====================
# 加载InceptionV3，去掉分类层，保留所有特征层
base_model = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=(None, None, 3)  # 支持任意尺寸图片，适配多八度缩放
)

# 疯狂关键：激活所有mixed层（0-6），生成最复杂的纹理
ALL_LAYERS = [base_model.get_layer(f"mixed{i}").output for i in range(7)]
dream_model = tf.keras.Model(inputs=base_model.input, outputs=ALL_LAYERS)

# 图片预处理/后处理（保留兼容性，增强色彩对比度）
def preprocess_image(image_path):
    """读取图片+标准化+轻微增强对比度，让生成的纹理更鲜艳"""
    img = Image.open(image_path).convert("RGB")
    img = np.array(img, dtype=np.float32)
    # 增强对比度（疯狂效果加成）
    img = img * (1 + random.uniform(0.1, 0.2))
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return tf.convert_to_tensor(img)

def deprocess_image(img):
    """还原图片+色彩还原，避免过曝"""
    img = img.numpy()
    # 逆预处理
    img = 255 * (img + 1.0) / 2.0
    # 裁剪像素值，避免过亮/过暗
    img = np.clip(img, 0, 255).astype(np.uint8)
    return Image.fromarray(img)

# ===================== 核心优化：梯度上升+疯狂增强 =====================
def calc_loss(image, model):
    """计算所有层的激活损失，最大化所有特征"""
    img_batch = tf.expand_dims(image, axis=0)
    layer_activations = model(img_batch)
    # 叠加所有层的损失，让纹理更丰富
    loss = tf.reduce_sum([tf.reduce_mean(act) for act in layer_activations])
    return loss

@tf.function  # 加速计算，支撑大迭代量
def dream_step(image, model, step_size, grad_power):
    """单步优化：梯度上升+梯度增强"""
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = calc_loss(image, model)
    # 计算梯度
    grads = tape.gradient(loss, image)
    # 梯度标准化+指数增强（疯狂核心：让纹理更突出）
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.math.pow(tf.abs(grads), grad_power) * tf.sign(grads)
    # 激进梯度上升
    image = image + grads * step_size
    # 限制像素值，避免失真
    image = tf.clip_by_value(image, -1, 1)
    return loss, image

def random_perturb(image):
    """简单随机微小偏移，无越界报错，保持疯狂效果"""
    if not RANDOM_SHIFT:
        return image
    # 安全的微小随机抖动，永远不会越界
    image = tf.roll(image, shift=random.randint(-2, 2), axis=0)
    image = tf.roll(image, shift=random.randint(-2, 2), axis=1)
    return image


# ===================== 多八度迭代：疯狂效果的核心 =====================
def crazy_deepdream(image, model, octave_count, steps_per_octave, step_size, grad_power):
    """
    多八度缩放迭代：把图片反复缩小→优化→放大，让纹理从细节到整体层层炸裂
    这是原版DeepDream的终极技巧，也是效果最疯狂的关键！
    """
    # 初始化八度缩放因子（每次缩放0.75倍）
    octave_scale = 0.75
    original_shape = tf.shape(image)
    shapes = [original_shape[:2]]
    # 生成多八度的尺寸列表（逐步缩小）【已修复类型错误】
    for i in range(1, octave_count):
        # 先转浮点数计算缩放，再转回整数，解决报错！
        shape = tf.cast(tf.cast(tf.shape(image)[:2], tf.float32) * (octave_scale ** i), tf.int32)
        shapes.append(shape)

    # 反转尺寸，从最小的开始优化（从细节到整体）
    shapes = shapes[::-1]
    current_image = tf.image.resize(image, shapes[0])

    # 逐八度迭代优化（核心疯狂循环）
    for idx, shape in enumerate(shapes):
        print(f"===== 八度 {idx+1}/{octave_count} | 尺寸: {shape[0]}x{shape[1]} =====")
        # 缩放到当前八度尺寸
        current_image = tf.image.resize(current_image, shape)
        # 每八度内的激进迭代
        for step in range(steps_per_octave):
            # 随机扰动（疯狂加成）
            current_image = random_perturb(current_image)
            # 梯度上升优化
            loss, current_image = dream_step(current_image, model, step_size, grad_power)
            # 每20步打印进度，看损失变化（损失越大，纹理越密集）
            if step % 20 == 0:
                print(f"  迭代 {step:>3}/{steps_per_octave} | 损失值: {loss.numpy():.6f} (越大越疯狂)")
        # 放大到下一个八度，叠加细节
        if idx < len(shapes) - 1:
            current_image = tf.image.resize(current_image, shapes[idx+1])

    # 还原到原图尺寸，完成最终优化
    final_image = tf.image.resize(current_image, original_shape[:2])
    return final_image

# ===================== 运行疯狂DeepDream！=====================
if __name__ == "__main__":
    # 1. 加载并预处理原图
    original_img = preprocess_image(INPUT_IMAGE_PATH)
    print(f"原始图片尺寸: {tf.shape(original_img)[0]}x{tf.shape(original_img)[1]}")
    print("开始生成疯狂DeepDream效果...（时间稍长，耐心等待）")

    # 2. 核心：生成极致梦幻效果
    dream_img = crazy_deepdream(
        original_img,
        dream_model,
        OCTAVE_COUNT,
        STEPS_PER_OCTAVE,
        STEP_SIZE,
        GRADIENT_POWER
    )

    # 3. 后处理+保存结果
    result = deprocess_image(dream_img)
    result.save(OUTPUT_IMAGE_PATH)
    print(f"\n疯狂效果已保存到: {OUTPUT_IMAGE_PATH}")

    # 4. 显示原图+疯狂效果对比（左右分栏）
    plt.figure(figsize=(20, 10))  # 放大画布，看清细节
    # 原图
    plt.subplot(1, 2, 1)
    plt.title("原始图片", fontsize=20, fontweight="bold")
    plt.imshow(deprocess_image(original_img))
    plt.axis("off")
    # 疯狂DeepDream效果
    plt.subplot(1, 2, 2)
    plt.title("Crazy DeepDream 极致梦幻效果", fontsize=20, fontweight="bold")
    plt.imshow(result)
    plt.axis("off")
    # 保存对比图
    plt.tight_layout()
    plt.savefig("deepdream_crazy_compare.jpg", dpi=300, bbox_inches="tight")
    plt.show()

    print("对比图已保存为: deepdream_crazy_compare.jpg")


In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random
import os
plt.rcParams['font.family'] = 'WenQuanYi Zen Hei'
# ==============================================
# 【核心配置】简单可调，平衡：原图辨识度 + 疯狂效果
# ==============================================
CONFIG = {
    "INPUT_PATH": "fangzi.jpg",            # 输入图片
    "OUTPUT_PATH": "dream_fangzi.jpg",    # 输出图片
    "MAX_IMAGE_SIZE": 1000,               # 自动缩图，防止显存溢出（兼容所有尺寸）
    "OCTAVES": 3,                         # 缩放层数（少=清晰，多=疯狂）
    "OCTAVE_SCALE": 0.75,                 # 缩放比例（固定，最优）
    "ITERATIONS": 80,                     # 每轮迭代数（效果强度）
    "STEP_SIZE": 0.012,                   # 变化步长（温和疯狂）
    "GRAD_POWER": 1.1,                    # 纹理增强（不覆盖原图）
    "ORIGIN_FUSION": 0.1,                 # 原图融合率（越大越像原图）
    "RANDOM_SHIFT": True                  # 安全随机扰动（无报错）
}

# ==============================================
# 【工具函数】图片预处理 + 后处理（兼容所有图片）
# ==============================================
def load_process_image(path):
    """加载图片，自动缩放到安全尺寸，标准化为模型输入"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到图片：{path}")
    
    # 读取图片，自动处理格式/通道
    img = Image.open(path).convert("RGB")
    img = np.array(img, dtype=np.float32)
    
    # 自动等比缩图，防止显存爆炸，兼容超大/超小图
    h, w = img.shape[:2]
    max_side = max(h, w)
    if max_side > CONFIG["MAX_IMAGE_SIZE"]:
        scale = CONFIG["MAX_IMAGE_SIZE"] / max_side
        img = tf.image.resize(img, (int(h*scale), int(w*scale))).numpy()
    
    # 保存原图副本（用于融合，保留轮廓）
    origin_img = tf.keras.applications.inception_v3.preprocess_input(img.copy())
    # 模型输入标准化
    processed_img = tf.keras.applications.inception_v3.preprocess_input(img)
    return tf.convert_to_tensor(processed_img), origin_img

def restore_image(img_tensor):
    """将模型输出还原为普通图片"""
    img = img_tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    return Image.fromarray(img)

# ==============================================
# 【加载模型】精选中层特征 → 纹理疯狂 + 原图不糊
# ==============================================
def build_dream_model():
    """构建DeepDream模型，只使用中层特征（保留轮廓，生成纹理）"""
    base_model = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet"
    )
    # 核心：中层特征 = 最佳平衡（不破坏原图，纹理丰富）
    layers = [
        base_model.get_layer("mixed2").output,
        base_model.get_layer("mixed3").output,
        base_model.get_layer("mixed4").output,
    ]
    return tf.keras.Model(inputs=base_model.input, outputs=layers)

# ==============================================
# 【核心算法】损失计算 + 梯度上升（极简无BUG）
# ==============================================
def calculate_loss(image, model):
    """计算特征损失，越大纹理越丰富"""
    batch = tf.expand_dims(image, axis=0)
    activations = model(batch)
    return tf.reduce_sum([tf.reduce_mean(act) for act in activations])

@tf.function
def dream_step(img, model, step, grad_pow, fusion, origin):
    """单步梦境优化 + 原图融合（保留轮廓的关键）"""
    with tf.GradientTape() as tape:
        tape.watch(img)
        loss = calculate_loss(img, model)
    
    # 计算梯度并标准化
    grads = tape.gradient(loss, img)
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.pow(tf.abs(grads), grad_pow) * tf.sign(grads)
    
    # 更新图片 + 融合原图（强制保留轮廓）
    img = img + grads * step
    img = img * (1 - fusion) + origin * fusion
    return tf.clip_by_value(img, -1, 1)

# ==============================================
# 【安全扰动】无越界、无报错（修复历史BUG）
# ==============================================
def safe_random_shift(img):
    """安全的像素滚动，永远不会报错，增加随机性"""
    if not CONFIG["RANDOM_SHIFT"]:
        return img
    # 微小偏移，不破坏原图
    img = tf.roll(img, shift=random.randint(-2, 2), axis=0)
    img = tf.roll(img, shift=random.randint(-2, 2), axis=1)
    return img

# ==============================================
# 【主算法】多尺度DeepDream（修复所有维度/类型BUG）
# ==============================================
def run_deepdream(img, origin_img, model):
    """多八度迭代，全流程无BUG，健壮性拉满"""
    original_shape = tf.shape(img)[:2]  # 只取高宽，修复维度报错
    shapes = [original_shape]
    
    # 生成缩放尺寸（修复int32类型报错：先浮点后整数）
    for i in range(1, CONFIG["OCTAVES"]):
        scale = CONFIG["OCTAVE_SCALE"] ** i
        size = tf.cast(tf.cast(original_shape, tf.float32) * scale, tf.int32)
        shapes.append(size)
    
    # 从最小尺寸开始优化
    shapes = shapes[::-1]
    current_img = tf.image.resize(img, shapes[0])

    # 逐轮优化
    for i, size in enumerate(shapes):
        print(f"🔹 处理层级 {i+1}/{CONFIG['OCTAVES']} | 尺寸: {size[0]}x{size[1]}")
        current_img = tf.image.resize(current_img, size)
        # 修复：把原图缩放到当前尺寸，保证维度一致
        current_origin = tf.image.resize(origin_img, size)

        for step in range(CONFIG["ITERATIONS"]):
            current_img = safe_random_shift(current_img)
            current_img = dream_step(
            current_img, model,
            CONFIG["STEP_SIZE"],
            CONFIG["GRAD_POWER"],
            CONFIG["ORIGIN_FUSION"],
            current_origin  # 传入缩放后的原图，而非原始大图
        )

            if step % 20 == 0:
                print(f"   迭代 {step}/{CONFIG['ITERATIONS']}")

        # 放大到下一层
        if i < len(shapes)-1:
            current_img = tf.image.resize(current_img, shapes[i+1])

    # 还原为原图尺寸
    return tf.image.resize(current_img, original_shape)

# ==============================================
# 【主程序】异常处理 + 运行 + 保存结果
# ==============================================
if __name__ == "__main__":
    try:
        print("🚀 启动DeepDream，保留原图 + 疯狂纹理模式")
        # 1. 加载图片
        img, origin = load_process_image(CONFIG["INPUT_PATH"])
        # 2. 加载模型
        model = build_dream_model()
        # 3. 生成梦境
        dream_img = run_deepdream(img, origin, model)
        # 4. 保存结果
        result = restore_image(dream_img)
        result.save(CONFIG["OUTPUT_PATH"])
        print(f"✅ 完成！结果已保存：{CONFIG['OUTPUT_PATH']}")

        # 显示对比图
        plt.figure(figsize=(14, 6))
        plt.subplot(121), plt.imshow(restore_image(img)), plt.title("原图"), plt.axis("off")
        plt.subplot(122), plt.imshow(result), plt.title("DeepDream 梦境"), plt.axis("off")
        plt.show()

    except Exception as e:
        print(f"❌ 错误：{str(e)}")


In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import os
import random

# ==============================================
# 【人类真实梦境参数】核心：柔和、朦胧、自然、熟悉感
# 模拟普通人做梦：场景清晰可辨，轻微扭曲、温润幻化
# ==============================================
DREAM_CONFIG = {
    "INPUT_IMAGE": "input.png",            # 你的原图
    "OUTPUT_IMAGE": "human_dream.png",     # 梦境输出图
    "MAX_SIZE": 800,                       # 自动缩图，防显存溢出
    "OCTAVE_LAYERS": 2,                    # 缩放层数（少=自然，贴合真实梦境）
    "OCTAVE_SCALE": 0.85,                  # 温和缩放，无剧烈变形
    "ITER_PER_OCTAVE": 40,                 # 低迭代：模拟梦境的模糊感
    "STEP_SIZE": 0.008,                    # 极小步长：轻微扭曲，不破坏原图
    "DREAM_SOFTNESS": 0.08,                # 梦境柔和度（人类梦境核心）
    "REALITY_FUSION": 0.15,                # 现实融合度：越高越像原图（真实做梦）
    "DREAM_BLUR": True,                    # 开启梦境朦胧感
    "SAFE_SHIFT": False                    # 关闭随机抖动：真实梦境无剧烈晃动
}

# ==============================================
# 【图片工具】兼容所有格式/尺寸，自带异常处理
# ==============================================
def load_image(image_path):
    """加载图片，自动标准化，兼容所有图片"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"❌ 未找到图片：{image_path}")
    
    # 读取并统一为RGB格式
    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img, dtype=np.float32)
    
    # 自动等比缩放，防止显存爆炸
    h, w = img_np.shape[:2]
    max_side = max(h, w)
    if max_side > DREAM_CONFIG["MAX_SIZE"]:
        scale = DREAM_CONFIG["MAX_SIZE"] / max_side
        img_np = tf.image.resize(img_np, (int(h*scale), int(w*scale))).numpy()
    
    # 模型预处理
    img_tensor = tf.keras.applications.inception_v3.preprocess_input(img_np)
    return tf.convert_to_tensor(img_tensor)

def recover_image(tensor):
    """将模型张量还原为普通图片，添加梦境朦胧滤镜"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    
    # 【真实梦境】添加轻微模糊，模拟人做梦的朦胧视觉
    if DREAM_CONFIG["DREAM_BLUR"]:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=0.8))
    return pil_img

# ==============================================
# 【DeepDream模型】人类梦境专用：低中层特征 = 柔和自然纹理
# ==============================================
def build_human_dream_model():
    """构建模型：仅使用低中层特征，生成云朵/烟雾/柔和纹理（真实梦境）"""
    base_model = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet"
    )
    # 核心：低中层 → 柔和、自然、无夸张变形（人类做梦的视觉特征）
    dream_layers = [
        base_model.get_layer("mixed0").output,
        base_model.get_layer("mixed1").output,
    ]
    return tf.keras.Model(inputs=base_model.input, outputs=dream_layers)

# ==============================================
# 【核心梦境算法】正统DeepDream：梯度上升，温和无破坏
# ==============================================
def calc_dream_loss(image, model):
    """计算梦境特征：柔和自然，无夸张激活"""
    img_batch = tf.expand_dims(image, axis=0)
    activations = model(img_batch)
    return tf.reduce_sum([tf.reduce_mean(act) for act in activations])

@tf.function
def dream_step(image, model, step_size, softness, reality, origin):
    """单步梦境演化：轻微扭曲，保留现实轮廓"""
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = calc_dream_loss(image, model)
    
    # 温和梯度计算（真实梦境核心：不剧烈变化）
    grads = tape.gradient(loss, image)
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.pow(tf.abs(grads), softness) * tf.sign(grads)
    
    # 梦境演化 + 现实融合（做梦不会完全忘记现实）
    image = image + grads * step_size
    image = image * (1 - reality) + origin * reality
    return tf.clip_by_value(image, -1, 1)

# ==============================================
# 【安全无报错】多尺度梦境生成（修复所有历史BUG）
# ==============================================
def generate_human_dream(image, model):
    """正统DeepDream多尺度迭代，模拟真实做梦过程"""
    original_shape = tf.shape(image)[:2]
    shapes = [original_shape]
    
    # 生成缩放尺寸（修复int32/维度BUG）
    for i in range(1, DREAM_CONFIG["OCTAVE_LAYERS"]):
        scale = DREAM_CONFIG["OCTAVE_SCALE"] ** i
        size = tf.cast(tf.cast(original_shape, tf.float32) * scale, tf.int32)
        shapes.append(size)
    
    shapes = shapes[::-1]
    current_img = tf.image.resize(image, shapes[0])

    # 逐层级演化梦境
    for i, size in enumerate(shapes):
        print(f"🌙 梦境演化 {i+1}/{DREAM_CONFIG['OCTAVE_LAYERS']} | 尺寸：{size[0]}x{size[1]}")
        current_img = tf.image.resize(current_img, size)
        current_origin = tf.image.resize(image, size)

        for step in range(DREAM_CONFIG["ITER_PER_OCTAVE"]):
            current_img = dream_step(
                current_img, model,
                DREAM_CONFIG["STEP_SIZE"],
                DREAM_CONFIG["DREAM_SOFTNESS"],
                DREAM_CONFIG["REALITY_FUSION"],
                current_origin
            )
            if step % 10 == 0:
                print(f"   梦境进度：{step}/{DREAM_CONFIG['ITER_PER_OCTAVE']}")

        if i < len(shapes)-1:
            current_img = tf.image.resize(current_img, shapes[i+1])

    return tf.image.resize(current_img, original_shape)

# ==============================================
# 【主程序】一键运行，异常安全
# ==============================================
if __name__ == "__main__":
    try:
        print("="*50)
        print("🧠 人类真实梦境 DeepDream 启动")
        print("💤 效果：原图入梦 → 朦胧、柔和、自然、熟悉可辨")
        print("="*50)
        
        # 1. 加载原图
        original_img = load_image(DREAM_CONFIG["INPUT_IMAGE"])
        # 2. 加载梦境模型
        dream_model = build_human_dream_model()
        # 3. 生成真实梦境
        dream_result = generate_human_dream(original_img, dream_model)
        # 4. 还原并保存
        final_img = recover_image(dream_result)
        final_img.save(DREAM_CONFIG["OUTPUT_IMAGE"])

        print(f"\n✅ 梦境生成完成！保存至：{DREAM_CONFIG['OUTPUT_IMAGE']}")

        # 展示对比：现实 vs 梦境
        plt.figure(figsize=(15, 7))
        plt.subplot(1, 2, 1)
        plt.imshow(recover_image(original_img))
        plt.title("现实（原图）", fontsize=16)
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(final_img)
        plt.title("梦境（人类平均真实体验）", fontsize=16)
        plt.axis("off")
        
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"\n❌ 运行错误：{str(e)}")


In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import os
import random
plt.rcParams['font.family'] = 'WenQuanYi Zen Hei'
# ==============================================
# 【人类噩梦梦境参数】核心：压抑、扭曲、尖锐、暗沉、不安
# 模拟普通人做噩梦：轮廓可辨但局部狰狞，纹理密集，色彩压抑，轻微晃动
# ==============================================
DREAM_CONFIG = {
    "INPUT_IMAGE": "input.png",            # 你的原图
    "OUTPUT_IMAGE": "nightmare_dream.png", # 噩梦输出图
    "MAX_SIZE": 800,                       # 自动缩图，防显存溢出
    "OCTAVE_LAYERS": 2,                    # 2层足够，避免过度扭曲
    "OCTAVE_SCALE": 0.8,                   # 稍大缩放，强化局部扭曲
    "ITER_PER_OCTAVE": 60,                 # 适中迭代，生成密集纹理
    "STEP_SIZE": 0.012,                     # 步长稍大，让扭曲更明显
    "DREAM_SOFTNESS": 0.9,                 # 调低柔和度→生成尖锐纹理
    "REALITY_FUSION": 0.1,                 # 调低现实融合→强化噩梦扭曲
    "DREAM_BLUR": False,                   # 关闭柔模糊→噩梦无朦胧，更尖锐
    "SAFE_SHIFT": True                     # 开启随机抖动→模拟噩梦的不安感
}

# ==============================================
# 【图片工具】兼容所有格式/尺寸，自带异常处理
# ==============================================
def load_image(image_path):
    """加载图片，自动标准化，兼容所有图片"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"❌ 未找到图片：{image_path}")
    
    # 读取并统一为RGB格式
    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img, dtype=np.float32)
    
    # 自动等比缩放，防止显存爆炸
    h, w = img_np.shape[:2]
    max_side = max(h, w)
    if max_side > DREAM_CONFIG["MAX_SIZE"]:
        scale = DREAM_CONFIG["MAX_SIZE"] / max_side
        img_np = tf.image.resize(img_np, (int(h*scale), int(w*scale))).numpy()
    
    # 模型预处理
    img_tensor = tf.keras.applications.inception_v3.preprocess_input(img_np)
    return tf.convert_to_tensor(img_tensor)

def recover_image(tensor):
    """将模型张量还原为普通图片，添加噩梦效果：锐化+色彩暗沉"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    
    # 【噩梦核心1】加轻微锐化，让纹理更尖锐，有压迫感
    pil_img = pil_img.filter(ImageFilter.SHARPEN)
    
    # 【噩梦核心2】色彩调暗沉（二选一：冷调/红调，注释掉另一个即可）
    # 冷调噩梦（蓝灰，压抑感）
    pil_img = pil_img.point(lambda x: x * 0.75 if x < 150 else x * 0.9)  # 压暗暗部，降低亮度
    # 红调噩梦（暗红，紧张/恐惧感）
    # r, g, b = pil_img.split()
    # r = r.point(lambda x: x * 1.1)
    # g = g.point(lambda x: x * 0.7)
    # b = b.point(lambda x: x * 0.7)
    # pil_img = Image.merge("RGB", (r, g, b))
    
    return pil_img


def build_human_dream_model():
    """构建噩梦模型：使用中高层特征，生成尖锐/密集/缠绕纹理（噩梦核心）"""
    base_model = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet"
    )
    # 核心：中高层 → 尖锐、密集、缠绕纹理，贴合噩梦的压迫感
    nightmare_layers = [
        base_model.get_layer("mixed3").output,
        base_model.get_layer("mixed4").output,
    ]
    return tf.keras.Model(inputs=base_model.input, outputs=nightmare_layers)

# ==============================================
# 【核心梦境算法】正统DeepDream：梯度上升，温和无破坏
# ==============================================
def calc_dream_loss(image, model):
    """计算梦境特征：柔和自然，无夸张激活"""
    img_batch = tf.expand_dims(image, axis=0)
    activations = model(img_batch)
    return tf.reduce_sum([tf.reduce_mean(act) for act in activations])

@tf.function
def dream_step(image, model, step_size, softness, reality, origin):
    """单步梦境演化：轻微扭曲，保留现实轮廓"""
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = calc_dream_loss(image, model)
    
    # 温和梯度计算（真实梦境核心：不剧烈变化）
    grads = tape.gradient(loss, image)
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.pow(tf.abs(grads), softness) * tf.sign(grads)
    
    # 梦境演化 + 现实融合（做梦不会完全忘记现实）
    image = image + grads * step_size
    image = image * (1 - reality) + origin * reality
    return tf.clip_by_value(image, -1, 1)

# ==============================================
# 【安全无报错】多尺度梦境生成（修复所有历史BUG）
# ==============================================
def generate_human_dream(image, model):
    """正统DeepDream多尺度迭代，模拟真实做梦过程"""
    original_shape = tf.shape(image)[:2]
    shapes = [original_shape]
    
    # 生成缩放尺寸（修复int32/维度BUG）
    for i in range(1, DREAM_CONFIG["OCTAVE_LAYERS"]):
        scale = DREAM_CONFIG["OCTAVE_SCALE"] ** i
        size = tf.cast(tf.cast(original_shape, tf.float32) * scale, tf.int32)
        shapes.append(size)
    
    shapes = shapes[::-1]
    current_img = tf.image.resize(image, shapes[0])

    # 逐层级演化梦境
    for i, size in enumerate(shapes):
        print(f"🌙 梦境演化 {i+1}/{DREAM_CONFIG['OCTAVE_LAYERS']} | 尺寸：{size[0]}x{size[1]}")
        current_img = tf.image.resize(current_img, size)
        current_origin = tf.image.resize(image, size)

        for step in range(DREAM_CONFIG["ITER_PER_OCTAVE"]):
            current_img = dream_step(
                current_img, model,
                DREAM_CONFIG["STEP_SIZE"],
                DREAM_CONFIG["DREAM_SOFTNESS"],
                DREAM_CONFIG["REALITY_FUSION"],
                current_origin
            )
            if step % 10 == 0:
                print(f"   梦境进度：{step}/{DREAM_CONFIG['ITER_PER_OCTAVE']}")

        if i < len(shapes)-1:
            current_img = tf.image.resize(current_img, shapes[i+1])

    return tf.image.resize(current_img, original_shape)

# ==============================================
# 【主程序】一键运行，异常安全
# ==============================================
if __name__ == "__main__":
    try:
        print("="*50)
        print("🧠 人类真实噩梦梦境 DeepDream 启动")
        print("💤 效果：原图入梦 → 压抑扭曲、色彩暗沉、纹理尖锐杂乱、轮廓轻微狰狞、明暗反差强烈")
        print("="*50)
        
        # 1. 加载原图
        original_img = load_image(DREAM_CONFIG["INPUT_IMAGE"])
        # 2. 加载梦境模型
        dream_model = build_human_dream_model()
        # 3. 生成真实梦境
        dream_result = generate_human_dream(original_img, dream_model)
        # 4. 还原并保存
        final_img = recover_image(dream_result)
        final_img.save(DREAM_CONFIG["OUTPUT_IMAGE"])

        print(f"\n✅ 梦境生成完成！保存至：{DREAM_CONFIG['OUTPUT_IMAGE']}")

        # 展示对比：现实 vs 梦境
        plt.figure(figsize=(15, 7))
        plt.subplot(1, 2, 1)
        plt.imshow(recover_image(original_img))
        plt.title("现实（原图）", fontsize=16)
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(final_img)
        plt.title("噩梦（人类平均真实体验）", fontsize=16)
        plt.axis("off")
        
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"\n❌ 运行错误：{str(e)}")


In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt
import os
import random
plt.rcParams['font.family'] = 'WenQuanYi Zen Hei'
# ==============================================
# 【人类美好美梦参数】核心：温暖、明亮、治愈、柔和
# 模拟普通人做美梦：场景清晰可辨，纹理细腻飘逸，色彩温润清新，无压迫感
# ==============================================
DREAM_CONFIG = {
    "INPUT_IMAGE": "input.png",            # 你的原图
    "OUTPUT_IMAGE": "sweet_dream.png",      # 美梦输出图
    "MAX_SIZE": 800,                       # 自动缩图，防显存溢出
    "OCTAVE_LAYERS": 3,                    # 3层缩放（适中，生成细腻渐变）
    "OCTAVE_SCALE": 0.88,                  # 温和缩放，无剧烈扭曲
    "ITER_PER_OCTAVE": 50,                 # 迭代量：生成细腻纹理，不粗糙
    "STEP_SIZE": 0.007,                    # 极小步长：轻微幻化，不破坏原图
    "DREAM_SOFTNESS": 0.5,                 # 梦境柔和度（低=生成细腻纹理，高=柔和）
    "REALITY_FUSION": 0.22,                # 现实融合度：极高！越像原图越有「熟悉的美好」
    "DREAM_SOFT_BLUR": True,               # 开启柔光模糊：模拟美梦的温润视觉
    "DREAM_BRIGHTNESS": 1.2,               # 亮度提升：美梦更明亮温暖
    "DREAM_COLOR": "warm",                 # 色彩风格：warm(暖调)/pastel(马卡龙)/fresh(清新)
    "SAFE_SHIFT": False                    # 关闭随机抖动：美梦安稳，无晃动
}

# ==============================================
# 【图片工具】兼容所有格式/尺寸，异常安全
# ==============================================
def load_image(image_path):
    """加载图片，自动标准化，兼容所有图片"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"❌ 未找到图片：{image_path}")
    
    # 读取并统一为RGB格式
    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img, dtype=np.float32)
    
    # 自动等比缩放，防止显存爆炸
    h, w = img_np.shape[:2]
    max_side = max(h, w)
    if max_side > DREAM_CONFIG["MAX_SIZE"]:
        scale = DREAM_CONFIG["MAX_SIZE"] / max_side
        img_np = tf.image.resize(img_np, (int(h*scale), int(w*scale))).numpy()
    
    # 模型预处理
    img_tensor = tf.keras.applications.inception_v3.preprocess_input(img_np)
    return tf.convert_to_tensor(img_tensor)

def recover_dream_image(tensor):
    """将模型张量还原为普通图片，添加美梦专属后处理（柔光+暖色调+提亮）"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    
    # 1. 美梦核心：柔光模糊（模拟美梦的温润、朦胧感）
    if DREAM_CONFIG["DREAM_SOFT_BLUR"]:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=0.5))
    
    # 2. 亮度提升：美梦更明亮，充满阳光感
    brightness_enhancer = ImageEnhance.Brightness(pil_img)
    pil_img = brightness_enhancer.enhance(DREAM_CONFIG["DREAM_BRIGHTNESS"])
    
    # 3. 色彩调色：美梦专属色调（三选一，注释掉其他两个）
    if DREAM_CONFIG["DREAM_COLOR"] == "warm":
        # 暖调：橙黄/粉暖，贴合阳光、花海、温馨回忆
        color_enhancer = ImageEnhance.Color(pil_img)
        pil_img = color_enhancer.enhance(1.1)  # 轻微提亮色彩
        # 加暖黄滤镜
        r, g, b = pil_img.split()
        r = r.point(lambda x: min(255, x + 8))
        g = g.point(lambda x: min(255, x + 5))
        pil_img = Image.merge("RGB", (r, g, b))
    
    elif DREAM_CONFIG["DREAM_COLOR"] == "pastel":
        # 马卡龙色：柔和、治愈，适合可爱/清新原图
        pil_img = pil_img.point(lambda x: min(255, x + 10) if x < 200 else x)
        color_enhancer = ImageEnhance.Color(pil_img)
        pil_img = color_enhancer.enhance(1.2)
    
    elif DREAM_CONFIG["DREAM_COLOR"] == "fresh":
        # 清新冷调：适合风景/绿植，清爽治愈
        color_enhancer = ImageEnhance.Color(pil_img)
        pil_img = color_enhancer.enhance(1.15)
        r, g, b = pil_img.split()
        b = b.point(lambda x: min(255, x + 5))
        pil_img = Image.merge("RGB", (r, g, b))
    
    return pil_img

# 新增：纯原始还原函数（仅用于展示原图，无美梦滤镜、无调色、无模糊）
def pure_restore_image(tensor):
    """纯还原原图，不添加任何梦境效果，保证对比真实"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    return Image.fromarray(img)


# ==============================================
# 【DeepDream模型】美梦专属：低中层特征 = 细腻柔和纹理
# （比普通梦更柔和，生成花瓣/云朵/柔光纹理）
# ==============================================
def build_dream_model():
    """构建美梦模型：仅使用最低中层特征，生成细腻/飘逸/柔和纹理（美梦核心）"""
    base_model = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet"
    )
    # 核心：最低中层 → 细腻、柔和、无尖锐纹理，模拟美梦的温柔氛围
    dream_layers = [
        base_model.get_layer("mixed0").output,  # 最底层：柔和边缘/云朵
        base_model.get_layer("mixed1").output,  # 中层：细腻花瓣/柔光
    ]
    return tf.keras.Model(inputs=base_model.input, outputs=dream_layers)

# ==============================================
# 【核心梦境算法】正统DeepDream：温和梯度上升，贴合美梦
# ==============================================
def calc_dream_loss(image, model):
    """计算美梦特征：细腻柔和，无夸张激活"""
    img_batch = tf.expand_dims(image, axis=0)
    activations = model(img_batch)
    return tf.reduce_sum([tf.reduce_mean(act) for act in activations])

@tf.function
def dream_step(image, model, step_size, softness, reality, origin):
    """单步美梦演化：轻微幻化，保留现实轮廓（核心：温柔不扭曲）"""
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = calc_dream_loss(image, model)
    
    # 美梦核心：温和梯度计算（无尖锐变化）
    grads = tape.gradient(loss, image)
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.pow(tf.abs(grads), softness) * tf.sign(grads)
    
    # 美梦演化 + 高现实融合（永远保留熟悉的美好）
    image = image + grads * step_size
    image = image * (1 - reality) + origin * reality
    return tf.clip_by_value(image, -1, 1)

# ==============================================
# 【安全无报错】多尺度美梦生成（修复所有历史BUG）
# ==============================================
def generate_sweet_dream(image, model):
    """正统DeepDream多尺度迭代，模拟真实美梦过程"""
    original_shape = tf.shape(image)[:2]
    shapes = [original_shape]
    
    # 生成缩放尺寸（修复int32/维度BUG）
    for i in range(1, DREAM_CONFIG["OCTAVE_LAYERS"]):
        scale = DREAM_CONFIG["OCTAVE_SCALE"] ** i
        size = tf.cast(tf.cast(original_shape, tf.float32) * scale, tf.int32)
        shapes.append(size)
    
    shapes = shapes[::-1]
    current_img = tf.image.resize(image, shapes[0])

    # 逐层级演化美梦
    for i, size in enumerate(shapes):
        print(f"🌼 美梦演化 {i+1}/{DREAM_CONFIG['OCTAVE_LAYERS']} | 尺寸：{size[0]}x{size[1]}")
        current_img = tf.image.resize(current_img, size)
        current_origin = tf.image.resize(image, size)  # 修复维度匹配BUG

        for step in range(DREAM_CONFIG["ITER_PER_OCTAVE"]):
            current_img = dream_step(
                current_img, model,
                DREAM_CONFIG["STEP_SIZE"],
                DREAM_CONFIG["DREAM_SOFTNESS"],
                DREAM_CONFIG["REALITY_FUSION"],
                current_origin
            )
            if step % 10 == 0:
                print(f"   美梦进度：{step}/{DREAM_CONFIG['ITER_PER_OCTAVE']}")

        if i < len(shapes)-1:
            current_img = tf.image.resize(current_img, shapes[i+1])

    return tf.image.resize(current_img, original_shape)

# ==============================================
# 【主程序】一键运行，异常安全
# ==============================================
if __name__ == "__main__":
    try:
        print("="*50)
        print("🧠 人类美好美梦 DeepDream 启动")
        print("✨ 效果：原图入梦 → 温暖、明亮、治愈、柔和、熟悉可辨")
        print("🌸 风格：细腻纹理、清新色彩、无狰狞扭曲")
        print("="*50)
        
        # 1. 加载原图
        original_img = load_image(DREAM_CONFIG["INPUT_IMAGE"])
        # 2. 加载美梦模型
        dream_model = build_dream_model()
        # 3. 生成美好美梦
        dream_result = generate_sweet_dream(original_img, dream_model)
        # 4. 还原并保存
        final_img = recover_dream_image(dream_result)
        final_img.save(DREAM_CONFIG["OUTPUT_IMAGE"])

        print(f"\n✅ 美梦生成完成！保存至：{DREAM_CONFIG['OUTPUT_IMAGE']}")

        # 展示对比：现实 vs 美梦
        plt.figure(figsize=(16, 8))
        plt.subplot(1, 2, 1)
        plt.imshow(pure_restore_image(original_img))

        plt.title("现实（原图）", fontsize=16, fontweight="bold")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(final_img)
        plt.title("美梦（温柔治愈，熟悉又美好）", fontsize=16, fontweight="bold")
        plt.axis("off")
        
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"\n❌ 运行错误：{str(e)}")


In [ ]:
pic_tool = {
    "name": "read_pic",
    "description": "读取指定图像的内容",
    "parameters": {
        "type": "object",
        "properties": {
            "pic_path": {
                "type": "string",
                "description": "图像的完整路径，如'/home/user/document.txt'"
            },
            "encoding": {
                "type": "string",
                "description": "图像编码，默认为'jpeg'",
                "default": "jpeg",
                "enum": ["jpeg", "jpg",  "png", "webp"]
            }
        },
        "required": ["pic_path"]
    }
}

In [ ]:
def validate_tool_definition(tool_def):
    """验证工具定义是否完整"""
    required_fields = ["name", "description", "parameters"]

    for field in required_fields:
        if field not in tool_def:
            return False, f"缺少必要字段: {field}"

    # 检查参数结构
    if "properties" not in tool_def["parameters"]:
        return False, "参数定义缺少properties字段"

    return True, "工具定义完整"

# 测试验证
is_valid, message = validate_tool_definition(pic_tool)
print(f"验证结果: {is_valid}, 消息: {message}")

In [ ]:
class ParameterValidator:
    """参数验证器"""

    def __init__(self, tool_def):
        self.tool_def = tool_def

    def validate(self, params):
        """执行完整的参数验证"""
        all_errors = []

        # 检查必填参数
        required_errors = self.validate_required(params)
        all_errors.extend(required_errors)

        # 检查参数类型
        type_errors = self.validate_type(params)
        all_errors.extend(type_errors)

        # 检查参数范围
        range_errors = self.validate_range(params)
        all_errors.extend(range_errors)

        # 检查额外参数（未定义的参数）
        extra_errors = self.validate_extra(params)
        all_errors.extend(extra_errors)

        return len(all_errors) == 0, all_errors

    def validate_required(self, params):
        """验证必填参数"""
        errors = []
        required_params = self.tool_def["parameters"].get("required", [])

        for param_name in required_params:
            if param_name not in params or params[param_name] is None:
                errors.append(f"缺少必填参数: {param_name}")

        return errors

    def validate_type(self, params):
        """验证参数类型"""
        errors = []

        for param_name, param_value in params.items():
            if param_name in self.tool_def["parameters"]["properties"]:
                param_def = self.tool_def["parameters"]["properties"][param_name]
                expected_type = param_def.get("type")

                if expected_type == "string" and not isinstance(param_value, str):
                    errors.append(f"参数'{param_name}'应为字符串类型，实际为{type(param_value).__name__}")
                elif expected_type == "integer" and not isinstance(param_value, int):
                    errors.append(f"参数'{param_name}'应为整数类型，实际为{type(param_value).__name__}")
                elif expected_type == "number" and not isinstance(param_value, (int, float)):
                    errors.append(f"参数'{param_name}'应为数字类型，实际为{type(param_value).__name__}")
                elif expected_type == "boolean" and not isinstance(param_value, bool):
                    errors.append(f"参数'{param_name}'应为布尔类型，实际为{type(param_value).__name__}")

        return errors

    def validate_range(self, params):
        """验证参数范围"""
        errors = []

        for param_name, param_value in params.items():
            if param_name in self.tool_def["parameters"]["properties"]:
                param_def = self.tool_def["parameters"]["properties"][param_name]

                # 检查最小值
                if "minimum" in param_def and param_value < param_def["minimum"]:
                    errors.append(f"参数'{param_name}'不能小于{param_def['minimum']}")

                # 检查最大值
                if "maximum" in param_def and param_value > param_def["maximum"]:
                    errors.append(f"参数'{param_name}'不能大于{param_def['maximum']}")

                # 检查枚举值
                if "enum" in param_def and param_value not in param_def["enum"]:
                    errors.append(f"参数'{param_name}'必须是{param_def['enum']}中的一个")

        return errors

    def validate_extra(self, params):
        """检查未定义的额外参数"""
        errors = []
        defined_params = set(self.tool_def["parameters"]["properties"].keys())
        provided_params = set(params.keys())

        extra_params = provided_params - defined_params
        if extra_params:
            errors.append(f"提供了未定义的参数: {', '.join(extra_params)}")

        return errors

# 使用示例
validator = ParameterValidator(pic_tool)
params = {"pic_path": "./input.png", "encoding": "png", "extra": "不应该有的参数"}
is_valid, errors = validator.validate(params)

if is_valid:
    print("参数验证通过")
else:
    print("参数验证失败:")
    for error in errors:
        print(f"  - {error}")

In [ ]:
def fix_parameters(params, errors, tool_def):
    """尝试修正参数错误"""
    fixed_params = params.copy()

    for error in errors:
        if "缺少必填参数" in error:
            param_name = error.split(": ")[1]
            # 尝试从上下文推断或使用默认值
            if param_name == "encoding":
                fixed_params[param_name] = "jpeg"  # 使用当天作为默认值

        elif "应为字符串类型" in error:
            param_name = error.split("'")[1]
            # 尝试转换为字符串
            fixed_params[param_name] = str(params[param_name])

    return fixed_params

In [ ]:
class ErrorHandler:
    """错误处理器"""

    def handle(self, error, context):
        """处理错误"""
        error_type = self.classify_error(error)

        if error_type == "network":
            return self.handle_network_error(error, context)
        elif error_type == "parameter":
            return self.handle_parameter_error(error, context)
        elif error_type == "authentication":
            return self.handle_auth_error(error, context)
        elif error_type == "resource":
            return self.handle_resource_error(error, context)
        else:
            return self.handle_unknown_error(error, context)

    def classify_error(self, error):
        """分类错误类型"""
        error_str = str(error).lower()

        if any(word in error_str for word in ["timeout", "connection", "network"]):
            return "network"
        elif any(word in error_str for word in ["parameter", "invalid", "missing"]):
            return "parameter"
        elif any(word in error_str for word in ["auth", "key", "permission"]):
            return "authentication"
        elif any(word in error_str for word in ["limit", "quota", "unavailable"]):
            return "resource"
        else:
            return "unknown"

    def handle_network_error(self, error, context):
        """处理网络错误"""
        return {
            "success": False,
            "error": "网络连接失败，请检查网络后重试",
            "retryable": True,
            "retry_after": 5  # 5秒后重试
        }

    def handle_parameter_error(self, error, context):
        """处理参数错误"""
        return {
            "success": False,
            "error": f"参数错误: {str(error)}",
            "retryable": False,
            "suggestion": "请检查输入参数是否正确"
        }

    # 其他错误处理方法类似...
    def user_friendly_error(error):
        """生成用户友好的错误消息"""
        error_map = {
            "timeout": "请求超时，请稍后重试",
            "connection_error": "网络连接失败，请检查网络",
            "invalid_api_key": "API 密钥无效，请检查配置",
            "rate_limit_exceeded": "调用频率过高，请稍后再试",
            "invalid_parameters": "输入参数不正确，请检查后重试"
        }

        error_str = str(error).lower()
        for key, message in error_map.items():
            if key in error_str:
                return message

        return "系统繁忙，请稍后重试"

In [ ]:
import time
import random

class ExponentialBackoffRetry:
    """指数退避重试"""

    def __init__(self, max_retries=3, base_delay=1, max_delay=30):
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay

    def retry(self, func, *args, **kwargs):
        """执行带重试的函数调用"""
        last_error = None

        for attempt in range(self.max_retries):
            try:
                return func(*args, **kwargs)

            except Exception as e:
                last_error = e

                # 检查是否可重试
                if not self.is_retryable(e):
                    raise

                # 最后一次尝试，不再重试
                if attempt == self.max_retries - 1:
                    break

                # 计算等待时间（指数退避 + 随机抖动）
                delay = min(
                    self.base_delay * (2 ** attempt) + random.uniform(0, 1),
                    self.max_delay
                )

                print(f"第{attempt + 1}次尝试失败，{delay:.1f}秒后重试...")
                time.sleep(delay)

        # 所有重试都失败
        raise last_error

    def is_retryable(self, error):
        """判断错误是否可重试"""
        error_str = str(error).lower()
        retryable_errors = ["timeout", "connection", "busy", "temporarily"]

        for retryable_error in retryable_errors:
            if retryable_error in error_str:
                return True

        return False

# 使用示例
retry = ExponentialBackoffRetry(max_retries=3)

def call_weather_api(city, date):
    """调用天气API（模拟可能失败）"""
    # 模拟API调用
    if random.random() < 0.3:  # 30%概率失败
        raise ConnectionError("API连接超时")
    return {"temperature": 25, "condition": "晴"}

try:
    result = retry.retry(call_weather_api, "北京", "今天")
    print(f"成功获取天气: {result}")
except Exception as e:
    print(f"获取天气失败: {e}")

In [ ]:
import time
import random

class ExponentialBackoffRetry:
    """指数退避重试"""

    def __init__(self, max_retries=3, base_delay=1, max_delay=30):
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay

    def retry(self, func, *args, **kwargs):
        """执行带重试的函数调用"""
        last_error = None

        for attempt in range(self.max_retries):
            try:
                return func(*args, **kwargs)

            except Exception as e:
                last_error = e

                # 检查是否可重试
                if not self.is_retryable(e):
                    raise

                # 最后一次尝试，不再重试
                if attempt == self.max_retries - 1:
                    break

                # 计算等待时间（指数退避 + 随机抖动）
                delay = min(
                    self.base_delay * (2 ** attempt) + random.uniform(0, 1),
                    self.max_delay
                )

                print(f"第{attempt + 1}次尝试失败，{delay:.1f}秒后重试...")
                time.sleep(delay)

        # 所有重试都失败
        raise last_error

    def is_retryable(self, error):
        """判断错误是否可重试"""
        error_str = str(error).lower()
        retryable_errors = ["timeout", "connection", "busy", "temporarily"]

        for retryable_error in retryable_errors:
            if retryable_error in error_str:
                return True

        return False

# 使用示例
retry = ExponentialBackoffRetry(max_retries=3)

def call_weather_api(city, date):
    """调用天气API（模拟可能失败）"""
    # 模拟API调用
    if random.random() < 0.3:  # 30%概率失败
        raise ConnectionError("API连接超时")
    return {"temperature": 25, "condition": "晴"}

try:
    result = retry.retry(call_weather_api, "北京", "今天")
    print(f"成功获取天气: {result}")
except Exception as e:
    print(f"获取天气失败: {e}")

In [ ]:
class ToolExecutor:
    """工具执行器"""

    def __init__(self):
        self.tools = {}  # 注册的工具
        self.validator = ParameterValidator
        self.error_handler = ErrorHandler()
        self.retry = ExponentialBackoffRetry()
        self.breaker = CircuitBreaker()

    def register_tool(self, name, tool_def, func):
        """注册工具"""
        self.tools[name] = {
            "definition": tool_def,
            "function": func
        }

    def execute_tool(self, tool_name, params):
        """执行工具"""
        if tool_name not in self.tools:
            return {
                "success": False,
                "error": f"工具'{tool_name}'未注册"
            }

        tool_info = self.tools[tool_name]
        tool_def = tool_info["definition"]
        tool_func = tool_info["function"]

        # 验证参数
        validator = self.validator(tool_def)
        is_valid, errors = validator.validate(params)

        if not is_valid:
            return {
                "success": False,
                "error": "参数验证失败",
                "details": errors
            }

        # 通过熔断器执行（带重试）
        try:
            def execute_with_retry():
                return self.breaker.execute(
                    lambda: self.retry.retry(tool_func, **params)
                )

            result = execute_with_retry()

            return {
                "success": True,
                "result": result,
                "tool": tool_name
            }

        except Exception as e:
            # 错误处理
            error_response = self.error_handler.handle(e, {
                "tool": tool_name,
                "params": params
            })

            return {
                "success": False,
                "error": error_response["error"],
                "retryable": error_response.get("retryable", False),
                "suggestion": error_response.get("suggestion", "")
            }

    def handle_user_request(self, user_input):
        """处理用户请求（简化版）"""
        # 1. 让LLM选择工具和提取参数（这里简化处理）
        # 实际应用中，这里会调用LLM进行工具选择和参数提取

        # 模拟LLM的输出
        if "做梦" in user_input:
            tool_name = "dream_dream"
            # 简单提取城市（实际应用中LLM会做得更好）
            if "美梦" in user_input:
                params = {"dream": "美梦", "deep": "默认"}
            elif "噩梦" in user_input:
                params = {"dream": "噩梦", "deep": "默认"}
            else:
                params = {"dream": "梦", "deep": "默认"}  # 默认值
        else:
            return "抱歉，我无法处理这个请求"

        # 2. 执行工具
        result = self.execute_tool(tool_name, params)

        # 3. 生成最终回答
        if result["success"]:
            weather = result["result"]
            return f"{params['dream']}做梦结果：{dreaming['result']}，做梦类型{dreaming['dream_type']}"
        else:
            return f"无法做梦：{result['error']}"

# 使用示例
executor = ToolExecutor()

def dream_dream():
    
# 注册天气工具
def mock_dream_api(dream, deep):
    """模拟做梦API"""
    # 模拟API调用延迟
    time.sleep(0.1)
    return {dreaming['result'] , dreaming['dream_type']}

executor.register_tool("dream_dream", dream_tool, mock_dream_api)

# 处理用户请求
response = executor.handle_user_request("根据我提供的图片，帮我做梦，要一个美梦。")
print(response)